# Inspect SageMaker Model Artifact

This notebook downloads a trained model (`model.tar.gz`) from S3, extracts the pickle file, loads it into memory, and displays detailed information about the model class, parameters, and feature importances.

In [ ]:
import os
import tarfile
import pickle
import shutil
import boto3
import numpy as np
import pandas as pd
from urllib.parse import urlparse

### 1. Configure S3 Path and Download Model

In [ ]:
# The exact S3 path of the trained model
model_s3_uri = "s3://aiconnex-cleaned/cmapss/v1/preprocessed/model/sagemaker-scikit-learn-2026-07-15-06-33-01-995/output/model.tar.gz"

parsed = urlparse(model_s3_uri)
bucket = parsed.netloc
key = parsed.path.lstrip('/')

local_dir = "./inspect_temp"
local_tar = os.path.join(local_dir, "model.tar.gz")

# Clean up previous runs if any
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
os.makedirs(local_dir, exist_ok=True)

print(f"Downloading model from s3://{bucket}/{key}...")
s3 = boto3.client('s3')
s3.download_file(bucket, key, local_tar)
print(f"Download complete. File size: {os.path.getsize(local_tar):,} bytes.")

### 2. Extract and Load the Pickle Model
*(This handles double-nested tarballs automatically)*

In [ ]:
extract_dir = os.path.join(local_dir, "extracted")
os.makedirs(extract_dir, exist_ok=True)

local_tmp = local_tar

# Peel back layers of nested tar.gz if they exist
for i in range(5):
    if not local_tmp.endswith(".tar.gz"):
        break
        
    print(f"Extracting layer {i+1}: {local_tmp}")
    layer_dir = os.path.join(extract_dir, f"layer_{i}")
    os.makedirs(layer_dir, exist_ok=True)
    
    with tarfile.open(local_tmp, "r:gz") as tar:
        tar.extractall(layer_dir)
        
    # Get files in the extracted folder
    extracted_files = []
    for root, _, files in os.walk(layer_dir):
        for f in files:
            extracted_files.append(os.path.join(root, f))
            
    pkl_files = [f for f in extracted_files if f.endswith(".pkl")]
    if pkl_files:
        local_tmp = pkl_files[0]
        break
        
    tar_files = [f for f in extracted_files if f.endswith(".tar.gz")]
    if tar_files:
        local_tmp = tar_files[0]
        continue
        
    raise ValueError(f"No pkl or nested tar found. Extracted: {extracted_files}")

print(f"Loading model into memory from: {local_tmp}")
with open(local_tmp, "rb") as f:
    model = pickle.load(f)
print("Model loaded successfully!")

### 3. Display Model Summary and Parameters

In [ ]:
print("=" * 60)
print("GENERAL METADATA:")
print(f"  Class Name: {type(model).__name__}")
print(f"  Module Name: {type(model).__module__}")

if hasattr(model, "n_estimators"):
    print(f"  Number of Trees: {model.n_estimators}")
if hasattr(model, "n_features_in_"):
    print(f"  Expected Input Features: {model.n_features_in_}")
print("=" * 60)

print("\nHYPEPARAMETERS / CONFIGURATION:")
for param, val in model.get_params().items():
    print(f"  {param}: {val}")

### 4. Analyze Feature Importances (Top 15)

In [ ]:
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    print("=" * 60)
    print("TOP 15 IMPORTANT FEATURES:")
    print("=" * 60)
    for i in range(min(15, len(importances))):
        feat_idx = indices[i]
        feat_val = importances[feat_idx]
        print(f"  Rank {i+1:2d} | Feature Index: {feat_idx:3d} | Importance: {feat_val:.6f}")
else:
    print("Model does not contain a feature_importances_ attribute.")